<a href="https://colab.research.google.com/github/ugoetudo/sequence-labeler/blob/main/TBOFullInference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, TFRobertaModel
import tensorflow as tf
import pandas as pd
import numpy as np
from google.colab import drive
from tqdm import tqdm
from tqdm.auto import tqdm
import keras
import random
from itertools import combinations
from keras import layers
from sklearn.model_selection import train_test_split
import json
from keras import regularizers
import spacy
!python -m spacy download 'en_core_web_md'
keras.config.disable_traceback_filtering()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 54.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
pd.options.display.max_columns = 80
pd.options.display.max_rows = 700

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base",
                                          add_prefix_space=True,
                                          clean_up_tokenization_spaces=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
NICKNAME = "breakthroughs_ner_tr_first_anno"
final_sequences = pd.read_csv(f"/content/drive/MyDrive/Colab Data/{NICKNAME}.csv")
final_sequences = final_sequences.rename(columns={"BIO_first":"target_BIO"})
NICKNAME = "ner_rand_all_types"
unique_tags = final_sequences['target_BIO'].unique().tolist()
tag2idx = {t: i for i, t in enumerate(unique_tags)}
idx2tag = {k:v for v,k in tag2idx.items()}
pad_tag = 72
idx2tag[pad_tag] = 'B-O'

In [6]:
@keras.saving.register_keras_serializable()
class TransformerBlock(layers.Layer):
  def __init__(self, embed_dim,
               num_heads,
               feed_forward_dim,
               max_len,
               dr_rate=0.0,
               train_mode=False,
               ):
    super().__init__()
    self.max_len = max_len
    self.train_mode = train_mode
    self.embed_dim = embed_dim
    self.num_heads = num_heads
    self.feed_forward_dim = feed_forward_dim
    self.dr_rate = dr_rate
    self.attention = layers.MultiHeadAttention(self.num_heads, key_dim=self.embed_dim)
    self.norm1 = layers.LayerNormalization(epsilon=1e-6)
    self.norm2 = layers.LayerNormalization(epsilon=1e-6)
    self.ff1 = layers.Dense(self.feed_forward_dim, activation="relu")
    self.ff2 = layers.Dense(self.embed_dim)
    self.dropout1 = layers.Dropout(self.dr_rate)
    self.dropout2 = layers.Dropout(self.dr_rate)

  def build(self, input_shape=None):
    if not self.ff1.built:
      self.ff1.build((self.max_len, self.embed_dim))
    if not self.ff2.built:
      self.ff2.build((self.max_len, self.feed_forward_dim))
    if not self.attention.built:
      self.attention.build([self.max_len, self.num_heads, self.embed_dim],
                           [self.max_len, self.num_heads, self.embed_dim])

  def get_config(self):

    return {"embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "feed_forward_dim": self.feed_forward_dim,
            "dr_rate": self.dr_rate,
            "train_mode":False}

  def call(self, inputs):
    attn_output = self.attention(*inputs)
    attn_output = self.dropout1(attn_output, training=self.train_mode)
    out1 = self.norm1(inputs[0] + attn_output)
    ffn_output = self.ff1(out1)
    ffn_output = self.ff2(ffn_output)
    ffn_output = self.dropout2(ffn_output, training=self.train_mode)
    ln2 = self.norm2(out1 + ffn_output)
    return ln2

@keras.saving.register_keras_serializable()
class robertaTransformerModel(keras.Model):
  def __init__(self,
               feed_forward_dim,
               num_tags,
               roberta_out="",
               train_mode=True,
               roberta_path="/content/drive/MyDrive/Colab Data/roberta-base",
               num_heads=5,
               max_len=80,
               **kwargs):
    super().__init__(**kwargs)
    self.max_len = max_len
    self.num_heads = num_heads
    self.roberta_path = roberta_path
    self.distilbert = TFRobertaModel.from_pretrained(self.roberta_path, name='roberta')
    self.roberta_out = roberta_out
    self.train_mode = train_mode
    self.num_tags = num_tags
    self.feed_forward_dim = feed_forward_dim
    self.dropout2 = layers.Dropout(0.2)
    self.transformer_block = TransformerBlock(self.distilbert.config.hidden_size,
                                              self.num_heads,
                                              self.distilbert.config.hidden_size,
                                              self.max_len,
                                              train_mode=self.train_mode)
    self.ffn = layers.Dense(self.feed_forward_dim, activation="relu", name="ffn")
    self.out = layers.Dense(self.num_tags, activation="softmax", name="out")

    self.distilbert.trainable = train_mode

  def build(self, input_shape=None):
    with tf.name_scope(self.distilbert.name):
      self.distilbert.build(None)
    if not self.transformer_block.built:
      self.transformer_block.build(input_shape=None)
    if not self.ffn.built:
      self.ffn.build((self.max_len, self.distilbert.config.hidden_size))
    if not self.out.built:
      self.out.build((self.max_len, self.feed_forward_dim))
    super().build(input_shape=None)

  def get_config(self):
    config = super().get_config()
    config.update({"feed_forward_dim": self.feed_forward_dim,
                   "num_tags": self.num_tags,
                   "train_mode": False,
                   "num_heads": self.num_heads,
                   "max_len": self.max_len,
                   "roberta_path": self.roberta_out})
    return config

  @classmethod
  def from_config(cls, config):
    return cls(**config)

  def call(self, inputs):
    encoded = self.distilbert(**inputs,
                        training=self.train_mode)
    x = encoded[0]
    if self.train_mode:
      x = self.dropout2(x)

    x = self.transformer_block((x,x))
    x = self.ffn(x) # last hidden state
    x = self.out(x)
    return x

  def save(self, filepath, overwrite=True, include_optimizer=True, save_format=None, signatures=None, options=None, **kwargs):

      print("Saving fine-tuned roberta model...")
      self.distilbert.save_pretrained(self.roberta_out)

      print("Saving keras model...")
      super().save(filepath, overwrite=True, **kwargs)

  def compile(self, model_optimizer, loss, metrics):
    super().compile(optimizer=model_optimizer, loss=loss, metrics=metrics)
    self.model_optimizer = model_optimizer
    self.loss = loss
    self.loss_metrics = metrics

  def get_compile_config(self):
    return {
        "model_optimizer": self.model_optimizer,
        "loss": self.loss,
        "metrics": self.loss_metrics
    }

  def compile_from_config(self, config):
    optimizer = keras.utils.deserialize_keras_object(config.get("model_optimizer"))
    loss = keras.utils.deserialize_keras_object(config.get("loss"))
    metrics = [keras.utils.deserialize_keras_object(metric) for metric in config.get("metrics")]
    self.compile(optimizer, loss, metrics)

In [7]:
def predict_from_string(test_sent, model, tokenizer, idx2tag):
  tokeinzed_sentence = tokenizer(test_sent, truncation=True,
            padding=True, max_length=saved_model.get_config()['max_len'],
            return_tensors="tf")
  decoded = tokenizer.convert_ids_to_tokens(tokeinzed_sentence['input_ids'][0])
  decoded = [tk[1:] if tk.find("Ġ") == 0 else tk for tk in decoded]
  pred = model.predict(dict(tokeinzed_sentence))
  pred = pred.argmax(axis=2)
  pred = [[idx2tag[i] for i in seq] for seq in pred]
  return pred, decoded, tokeinzed_sentence

In [8]:
def normalize_predictions(pred, decoded, tk_sent, single_pred=True):
  wids = tk_sent.word_ids()
  recon = []
  curr_pred = []
  last_wid = None
  for i in range(len(wids)):
    if wids[i] is None:
      last_wid = 1
      continue
    if last_wid is None:
      last_wid = wids[i]
    if last_wid != wids[i]:
      recon.append(decoded[i])
      curr_pred.append([pred[0][i]])
      last_wid = wids[i]
    else:
      last_recon = recon.pop()
      recon.append(last_recon + decoded[i])
      curr_pred[-1].append(pred[0][i])
      last_wid = wids[i]
  if single_pred:
    curr_pred = [list(set(item))[0] for item in curr_pred]
    curr_pred = ["O" if item == "B-O" else item for item in curr_pred]
  else:
    curr_pred = [list(set(item)) for item in curr_pred]
  assert len(curr_pred) == len(recon)
  return curr_pred, recon

In [9]:
# Credit to Priyankar Bose for the code in this cell

def setup_relationship_extraction(tokens, entity_tags):
    # Ensuring that the length of tokens and entities match
    if len(tokens) != len(entity_tags):
        raise ValueError("The lengths of tokens and entity_tags must match.")

    relationship_data = []
    sentence_text = " ".join(tokens).replace("<s> ", "")  # Ignoring <s> (space)
    # Generating a random sentence ID
    sentence_id = random.randint(1000, 9999)

    # Generating token IDs array starting from 0
    tkids = list(range(len(tokens)))

    # Nested function to extract entities from tokens and tags
    def extract_entities(tokens, tags, tkids):
        entities = []
        current_entity = []
        current_entity_type = None
        current_entity_ids = []

        for token, tag, tkid in zip(tokens, tags, tkids):
            if tag.startswith("B-") or (tag.startswith("I-") and current_entity_type != tag[2:]) :  # Beginning of a new entity
                if current_entity:  # Save the previous entity
                    entities.append((" | ".join(map(str, current_entity_ids)), current_entity_type, " ".join(current_entity)))
                # Start a new entity
                current_entity = [token]
                current_entity_type = tag[2:]  # Entity type
                current_entity_ids = [tkid]
            elif tag.startswith("I-") and current_entity_type == tag[2:]:  # Continuation of the same entity
                current_entity.append(token)
                current_entity_ids.append(tkid)
            elif tag == "O" and current_entity:  # Ignoring "O" but continue the current entity
                continue
            else:  # Outside of an entity
                if current_entity:  # Save the previous entity
                    entities.append((" | ".join(map(str, current_entity_ids)), current_entity_type, " ".join(current_entity)))
                current_entity = []
                current_entity_type = None
                current_entity_ids = []

        # Add the last entity if present
        if current_entity:
            entities.append((" | ".join(map(str, current_entity_ids)), current_entity_type, " ".join(current_entity)))

        return entities

    # Extracting entities
    entities = extract_entities(tokens, entity_tags, tkids)

    # Generating all combinations of two entities with direction (left, right)
    for left, right in combinations(entities, 2):
        left_entity_ids, left_entity_type, left_full_text = left
        right_entity_ids, right_entity_type, right_full_text = right

        # Creating a relationship entry for left-to-right direction
        relationship_data.append({
            "sid": sentence_id,
            "lefttkids": left_entity_ids,
            "righttkids": right_entity_ids,
            "sentence_text": sentence_text,
            "left_entity": left_entity_type,
            "lefttext": left_full_text,
            "right_entity": right_entity_type,
            "righttext": right_full_text
        })

        # Creating a relationship entry for right-to-left direction
        relationship_data.append({
            "sid": sentence_id,
            "lefttkids": right_entity_ids,
            "righttkids": left_entity_ids,
            "sentence_text": sentence_text,
            "left_entity": right_entity_type,
            "lefttext": right_full_text,
            "right_entity": left_entity_type,
            "righttext": left_full_text
        })

    return relationship_data

In [10]:
MAX_ENTITY_LENGTH = 5
MAX_SENTENCE_LEN = 100
with open("/content/drive/MyDrive/Colab Data/entity_lookup_rels.json", "r") as ent_lookup_file:
  entity_lookup = json.load(ent_lookup_file)

def make_pos_embs_l(x):
  entity_vec = np.zeros(MAX_SENTENCE_LEN)
  tkids = str(x['lefttkids']).replace(" ", "").split("|")
  tkids = [int(tk) for tk in tkids]
  entity_vec[tkids] = entity_vec[tkids] + entity_lookup[x['left_entity']]
  entity_vec[0] = 1 #encodes the left or the right
  return entity_vec

def make_pos_embs_r(x):
  entity_vec = np.zeros(MAX_SENTENCE_LEN)
  tkids = str(x['righttkids']).replace(" ", "").split("|")
  tkids = [int(tk) for tk in tkids]
  entity_vec[tkids] = entity_vec[tkids] + entity_lookup[x['right_entity']]
  entity_vec[0] = 2 #encodes the left or the right
  return entity_vec

def get_entity_start_l(x):
  tkids = str(x['lefttkids']).replace(" ", "").split("|")
  tkids = [int(tk) for tk in tkids]
  rtkids = str(x['righttkids']).replace(" ", "").split("|")
  rtkids = [int(tk) for tk in rtkids]
  if rtkids[0] > tkids[0]:
    return tkids[0]
  else:
    return tkids[-1]

def get_entity_start_r(x):
  tkids = str(x['righttkids']).replace(" ", "").split("|")
  tkids = [int(tk) for tk in tkids]
  ltkids = str(x['lefttkids']).replace(" ", "").split("|")
  ltkids = [int(tk) for tk in ltkids]
  if ltkids[0] > tkids[0]:
    return tkids[0]
  else:
    return tkids[-1]

def slice_sentence(x):
  ls = x["l_entity_start"]
  rs = x["r_entity_start"]
  if rs > ls:
    s_slice = x["sentence_text"][ls:rs]
  else:
    s_slice = x["sentence_text"][rs:ls]
  return s_slice


def build_sentence_input(sentence_text):
  tokenized_sentence = tokenizer(sentence_text,
                                 max_length=MAX_SENTENCE_LEN,
                                 truncation=True,
                                 padding="max_length",
                                 is_split_into_words=True,
                                 return_tensors="tf"
                                 )
  # input_ids = tokenized_sentence["input_ids"]
  # attention_mask = tokenized_sentence["attention_mask"]
  return tokenized_sentence




In [11]:
@keras.saving.register_keras_serializable()
class TransformerBlock(layers.Layer):
  def __init__(self,
               embed_dim,
               num_heads,
               feed_forward_dim,
               max_len,
               dr_rate=0.25,
               train_mode=False):
    super().__init__()
    self.max_len = max_len
    self.embed_dim = embed_dim
    self.train_mode = train_mode
    self.num_heads = num_heads
    self.feed_forward_dim = feed_forward_dim
    self.dr_rate = dr_rate
    self.attention = layers.MultiHeadAttention(self.num_heads, key_dim=self.embed_dim)
    self.norm1 = layers.LayerNormalization(epsilon=1e-6)
    self.norm2 = layers.LayerNormalization(epsilon=1e-6)
    self.ff1 = layers.Dense(self.feed_forward_dim, activation="relu")
    self.ff2 = layers.Dense(self.embed_dim)
    self.dropout1 = layers.Dropout(self.dr_rate)
    self.dropout2 = layers.Dropout(self.dr_rate)

  def build(self, input_shape=None):
    if not self.ff1.built:
      self.ff1.build((self.max_len, self.embed_dim))
    if not self.ff2.built:
      self.ff2.build((self.max_len, self.feed_forward_dim))
    if not self.attention.built:
      self.attention.build([self.max_len, self.num_heads, self.embed_dim],
                           [self.max_len, self.num_heads, self.embed_dim])

  def get_config(self):

    return {"embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "feed_forward_dim": self.feed_forward_dim,
            "dr_rate": self.dr_rate}

  def call(self, inputs):
    attn_output = self.attention(*inputs)
    attn_output = self.dropout1(attn_output, training=self.train_mode)
    out1 = self.norm1(inputs[0] + attn_output)
    ffn_output = self.ff1(out1)
    ffn_output = self.ff2(ffn_output)
    ffn_output = self.dropout2(ffn_output, training=self.train_mode)
    ln2 = self.norm2(out1 + ffn_output)
    return ln2

@keras.saving.register_keras_serializable()
class valenceIntegratedModel(keras.Model):
  def __init__(self,
               feed_forward_dim,
               pos_emb_dim,
               rnn_hidden, #
               num_tags,
               num_valence,
               max_len,
               num_heads,
               train_mode,
               num_ents,
               dropout_rate=0.1,
               roberta_out="/content/drive/MyDrive/Colab Data/vire_fine_tuned/roberta_vire_embeddings",
               roberta_slice_out="/content/drive/MyDrive/Colab Data/vire_fine_tuned/roberta_slice_vire_embeddings",
               roberta_path="/content/drive/MyDrive/Colab Data/roberta-base",
               roberta_slice_path="/content/drive/MyDrive/Colab Data/roberta-base",
               **kwargs):
    super().__init__(**kwargs)
    self.training = train_mode
    self.dropout_rate = dropout_rate
    self.max_len = max_len
    self.train_mode = train_mode
    self.num_ents = num_ents
    self.feed_forward_dim = feed_forward_dim
    self.pos_emb_dim = pos_emb_dim
    self.rnn_hidden = rnn_hidden
    self.num_tags = num_tags
    self.num_heads = num_heads
    self.num_valence = num_valence
    self.roberta_path = roberta_path
    self.roberta_out = roberta_out
    self.roberta_slice_out = roberta_slice_out
    self.roberta = TFRobertaModel.from_pretrained(roberta_path,
                                                     trainable=self.train_mode,
                                                  name='roberta')
    self.roberta_slice = TFRobertaModel.from_pretrained(roberta_slice_path,
                                                        trainable=self.train_mode,
                                                        name='roberta_slice')
    self.droupout = layers.Dropout(self.dropout_rate)
    self.pos = layers.TimeDistributed(\
                              layers.Embedding(input_dim=num_ents,
                                               embeddings_regularizer=regularizers.L2(),
                                               output_dim=self.pos_emb_dim,
                                               trainable=self.train_mode))
    self.perm = layers.Permute((2,1,3))
    self.reshape = layers.Reshape((self.max_len, self.pos_emb_dim*2))
    self.dropout1 = layers.Dropout(self.dropout_rate)
    self.roberta_out = roberta_out
    self.transformer_block = TransformerBlock(self.roberta.config.hidden_size,
                                              self.num_heads,
                                              self.roberta.config.hidden_size,
                                              self.max_len,
                                              train_mode=self.train_mode,
                                              dr_rate=self.dropout_rate)
    self.pre_encoder = layers.Dense(self.roberta.config.hidden_size)
    self.rel_enc_trans = TransformerBlock(self.roberta.config.hidden_size,
                                              self.num_heads,
                                              self.roberta.config.hidden_size,
                                              self.max_len,
                                              train_mode=self.train_mode,
                                              dr_rate=self.dropout_rate)
    self.gmp = layers.GlobalMaxPooling1D()
    self.dnn_first = layers.Dense(self.feed_forward_dim, activation="relu")
    self.dnn_sec = layers.Dense(self.feed_forward_dim, activation="relu")
    self.dnn_third = layers.Dense(self.feed_forward_dim, activation="relu")
    self.out = layers.Dense(self.num_tags,
                            activation="softmax",
                            name='rel_out')
    self.valence_dnn1 = layers.Dense(self.feed_forward_dim,
                                     activation="relu")
    self.valence_out = layers.Dense(self.num_valence,
                                    activation="softmax",
                                    name='valence_out')

  def build(self):
    if not self.pos.built:
      self.pos.build(input_shape=(2, self.max_len, num_ents))
    with tf.name_scope(self.roberta.name):
      self.roberta.build(None)
    with tf.name_scope(self.roberta_slice.name):
      self.roberta_slice.build(None)
    if not self.transformer_block.built:
      self.transformer_block.build(input_shape=None)
    if not self.pre_encoder.built:
      self.pre_encoder.build((self.max_len,
                              self.roberta.config.hidden_size+(self.pos_emb_dim*2)))
    if not self.rel_enc_trans.built:
      self.rel_enc_trans.build(input_shape=None)
    if not self.dnn_first.built:
      self.dnn_first.build((1, self.roberta.config.hidden_size*2))
    if not self.dnn_sec.built:
      self.dnn_sec.build((1, self.feed_forward_dim))
    if not self.dnn_third.built:
      self.dnn_third.build((1, self.feed_forward_dim))
    if not self.out.built:
      self.out.build((1, self.feed_forward_dim))
    if not self.valence_dnn1.built:
      self.valence_dnn1.build((1, self.num_tags+self.feed_forward_dim))
    if not self.valence_out.built:
      self.valence_out.build((1, self.feed_forward_dim))
    # super().build(input_shape=None)

  def get_config(self):
    config = super().get_config()
    config.update({
        "feed_forward_dim": self.feed_forward_dim, # hyperopt
        "pos_emb_dim": self.pos_emb_dim, # hyperopt
        "rnn_hidden": self.rnn_hidden, # not used
        "num_tags": self.num_tags,
        "num_valence": self.num_valence,
        "num_ents": self.num_ents,
        "max_len": self.max_len,
        "dropout_rate": self.dropout_rate,
        "num_heads": self.num_heads, # hyperopt
        "train_mode": False,
        "roberta_out": self.roberta_out,
        "roberta_slice_out": self.roberta_slice_out,
        "roberta_path": self.roberta_out,
        "roberta_slice_path": self.roberta_slice_out,
    })
    return config

  @classmethod
  def from_config(cls, config):
    return cls(**config)

  def save(self, filepath, overwrite=True,
           include_optimizer=True, save_format=None,
           signatures=None, options=None, **kwargs):

      print("Saving fine-tuned roberta models...")
      self.roberta.save_pretrained(self.roberta_out)
      self.roberta_slice.save_pretrained(self.roberta_slice_out)
      print("Saving keras model...")
      super().save(filepath, overwrite=True, **kwargs)

  def call(self, inputs):
    # Begin relation encoder
    roberta_output = self.roberta(**inputs[0],
                                     training=self.train_mode)
    pos_embs = tf.stack([inputs[1], inputs[2]], axis=1)
    pos_embs = self.pos(pos_embs)
    pos_embs = self.perm(pos_embs)
    pos_embs = self.reshape(pos_embs)
    enc = layers.concatenate([roberta_output[0],
                              pos_embs],
                              axis=-1) #along the final axis
    enc = self.pre_encoder(enc)
    rnn_output = self.rel_enc_trans((enc, enc))
    if self.train_mode:
      enc = self.droupout(enc)
    # End relation encoder
    # Begin inner-context encoder
    enc_slice = self.roberta_slice(**inputs[3],
                                   training=self.train_mode)
    context_embeddings = self.transformer_block((enc_slice[0], enc))
    # End inner-context encoder
    inner_context = self.gmp(context_embeddings)
    rnn_output = self.gmp(rnn_output)
    rnn_output = layers.concatenate([rnn_output, inner_context], axis=-1)
    feed_forward_output = self.dnn_first(rnn_output)
    feed_forward_output = self.dnn_sec(feed_forward_output)
    feed_forward_output = self.dnn_third(feed_forward_output)
    x = self.out(feed_forward_output)
    valence_out = self.valence_dnn1(layers.concatenate([x, feed_forward_output]))
    valence_out = self.valence_out(valence_out)
    return {"rel_out":x, "valence_out":valence_out}

In [12]:
NICKNAME = "ner_rand_all_types"
saved_model = keras.models.load_model(f"/content/drive/MyDrive/Colab Data/{NICKNAME}.keras", compile=False)
saved_model.summary()

All model checkpoint layers were used when initializing TFRobertaModel.

All the layers of TFRobertaModel were initialized from the model checkpoint at /content/drive/MyDrive/Colab Data/roberta_ner_embeddings_ner_rand_all_types.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


Model: "roberta_transformer_model_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block (TransformerBlock) │ ?                           │      29,521,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ ffn (Dense)                          │ (80, 256)                   │         196,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ out (Dense)                          │ (80, 73)                    │          18,761 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 29,736,777 (113.44 MB)

 Trainable params: 29,736,777 (113.44 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
num_ents = len(entity_lookup.keys())+1
m = keras.models.load_model("/content/drive/MyDrive/Colab Data/valenced_re_integrated_att.keras", compile=False)
m.summary()

All model checkpoint layers were used when initializing TFRobertaModel.

All the layers of TFRobertaModel were initialized from the model checkpoint at /content/drive/MyDrive/Colab Data/vire_fine_tuned/roberta_vire_embeddings.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.
All model checkpoint layers were used when initializing TFRobertaModel.

All the layers of TFRobertaModel were initialized from the model checkpoint at /content/drive/MyDrive/Colab Data/vire_fine_tuned/roberta_slice_vire_embeddings.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


Model: "valence_integrated_model_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dropout_4 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ (2, 100, 67, 128)           │           8,576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ permute (Permute)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_1                  │ ?                           │      20,074,752 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (100, 768)                  │         787,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_2                  │ ?                           │      20,074,752 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (1, 256)                    │         393,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (1, 256)                    │          65,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (1, 256)                    │          65,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rel_out (Dense)                      │ (1, 18)                     │           4,626 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (1, 256)                    │          70,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ valence_out (Dense)                  │ (1, 4)                      │           1,028 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 41,546,390 (158.49 MB)

 Trainable params: 41,537,814 (158.45 MB)

 Non-trainable params: 8,576 (33.50 KB)

In [36]:
from spacy.tokens.doc import Doc
nlp = spacy.load("en_core_web_md", disable=["lemmatizer", "ner"])
def spacy_rbt_tokenizer(text):
  tokeinzed_sentence = tokenizer(text, truncation=True,
            padding=True, max_length=saved_model.get_config()['max_len'],
            return_tensors="tf")
  decoded = tokenizer.convert_ids_to_tokens(tokeinzed_sentence['input_ids'][0])
  decoded = [tk[1:] if tk.find("Ġ") == 0 else tk for tk in decoded]
  decoded.pop()
  decoded = decoded[1:]
  return Doc(nlp.vocab, decoded)
nlp.tokenizer = spacy_rbt_tokenizer


In [24]:

def do_ner(test_sent, saved_model, tokenizer, idx2tag):
  pred, decoded, tk_sent = predict_from_string(test_sent, saved_model, tokenizer, idx2tag)
  norm_preds, recon_sent = normalize_predictions(pred, decoded, tk_sent)
  ner_viz = pd.DataFrame([norm_preds, recon_sent])
  relationship_data = setup_relationship_extraction(recon_sent, norm_preds)
  s_all_relations = pd.DataFrame(relationship_data)
  return s_all_relations, ner_viz

In [15]:
def do_relation_extraction(s_all_relations):
  s_all_relations["sentence_text"] = s_all_relations["sentence_text"].str.split(" ")
  s_all_relations['num_tokens'] = s_all_relations['sentence_text'].apply(len)
  s_all_relations["l_pos_embs"] = s_all_relations.apply(make_pos_embs_l, axis=1)
  s_all_relations["r_pos_embs"] = s_all_relations.apply(make_pos_embs_r, axis=1)
  s_all_relations["l_entity_start"] = s_all_relations.apply(get_entity_start_l, axis=1)
  s_all_relations["r_entity_start"] = s_all_relations.apply(get_entity_start_r, axis=1)
  s_all_relations["inner_context"] = s_all_relations.apply(slice_sentence, axis=1)
  l_pos_embs = tf.convert_to_tensor(np.stack(s_all_relations['l_pos_embs'].values))
  r_pos_embs = tf.convert_to_tensor(np.stack(s_all_relations['r_pos_embs'].values))
  sent_input = build_sentence_input(s_all_relations["sentence_text"].tolist())
  inner_context_input = build_sentence_input(s_all_relations["inner_context"].tolist())
  targets = ['Present', 'none', 'Cooperates', 'Informs', 'Is',
           'Opposes', 'Agrees', 'Threat', 'Belongs',
           'MovesFor', 'Thinks', 'Caused', 'IsNot', 'Obtains',
           'Disagrees', 'Influences', 'Harms', 'Kills']
  rel_target_encoding = {target: i for i, target in enumerate(targets)}

  v_target_encoding = {v_target: i for i, v_target in enumerate(['Neutral',
                                                                'Negative',
                                                                'Positive',
                                                                'none'])}
  cr_targets = {v:k for k,v in rel_target_encoding.items()}
  cv_targets = {v:k for k, v in v_target_encoding.items()}
  lps = m.predict([
                    dict(sent_input),
                    l_pos_embs,
                    r_pos_embs,
                    dict(inner_context_input)
                  ],
                  batch_size=32)
  vp = lps['valence_out'].argmax(axis=1)
  rp = lps['rel_out'].argmax(axis=1)

  s_all_relations['pred_rel'] = rp
  s_all_relations['pred_rel'] = s_all_relations['pred_rel'].map(cr_targets)
  s_all_relations['pred_val'] = vp
  s_all_relations['pred_val'] = s_all_relations['pred_val'].map(cv_targets)
  s_all_relations = s_all_relations.sort_values(by='sid')[["sid","sentence_text",
                                                           "lefttext","righttext",
                                                           "pred_rel", "pred_val"]]
  return s_all_relations

In [16]:
def get_nodes_from_batch(sents_batch):
  all_nodes = pd.DataFrame(columns=["sid", "sentence_text", "lefttext", "righttext", "pred_rel", "pred_val"])
  ner_visuals = pd.DataFrame(columns=[f"t{i}" for i in range(80)])
  for test_sent in sents_batch:
    s_all_relations, ner_visual = do_ner(test_sent, saved_model, tokenizer, idx2tag)
    ner_visual = ner_visual.rename(columns={i:f"t{i}" for i in range(80)})
    if s_all_relations.shape[0] > 0:
      extracted_nodes = do_relation_extraction(s_all_relations)
    else:
      print("No Relations Found")
      extracted_nodes = pd.DataFrame(columns=["sid", "sentence_text", "lefttext", "righttext", "pred_rel", "pred_val"])
    all_nodes = pd.concat([all_nodes, extracted_nodes])
    ner_visuals = pd.concat([ner_visuals, ner_visual])
  return all_nodes, ner_visuals

In [17]:
test_sents = ["Elon Musk is an overt white supremacy and nazism from a fascist who grew up in apartheid South Africa.",
"Tariffs on Canada, Mexico and China are a cudgel to force those countries, America's largest trading partners, to crack down on the flow of drugs and migrants into the United States.",
"Russian oligarchs have hidden their money in London for twenty years, encouraged by British bankers, lawyers and politicians.",
"Surely you could have picked a pundit who hasn't been parroting white supremacist talking points so much that he's Richard Spencer's favorite TV personality.",
"Amazon pulled out of New York before they pulled out of advertising on Breitbart",
"Shopify has blocked us for pointing out that he helps sites like Breitbart monetize bigotry by selling their merchandise and his argument for doing it is free speech.",
"Very strange what happens when you bag on diversity and inclusion every night when your show is sponsored by companies that have pages on their websites touting diversity and inclusion.",]

In [18]:
raw_sample = pd.read_csv('/content/drive/MyDrive/Colab Data/sg_tl_sample.csv')
raw_sample['samples'] = raw_sample['samples'].str.replace("Ã¢â‚¬â„¢", "'")
raw_sample['samples'] = raw_sample['samples'].str.replace("Ã¢â‚¬Å“", "'")
raw_sample['samples'] = raw_sample['samples'].str.replace("Ã¢â‚¬Â", "'")
raw_sample['samples'] = raw_sample['samples'].str.replace("http", " http")
raw_sample['samples'] = raw_sample['samples'].str.replace("Ã¢Â\x81Â¦", "")
raw_sample['samples'] = raw_sample['samples'].str.replace("Ã¢Â\x81Â©", "")
raw_sample['samples'] = raw_sample['samples'].str.replace("\x9d", "")
raw_sample['samples'] = raw_sample['samples'].str.split("http")
raw_sample['samples'] = raw_sample['samples'].apply(lambda x: x[0])
raw_sample['samples'] = raw_sample['samples'].str.split("pic.")
raw_sample['samples'] = raw_sample['samples'].apply(lambda x: x[0])
test_sents = raw_sample['samples'].values.tolist()
test_sents

["That is your prerogative if you want to support him, but there are plenty of other great people who will fight for everyone's rights who haven't ever considered wearing blackface.",
 "None of these dudes can ever believe they're being called on their shit.",
 'Everyone deserves to be given a chance to change, but how can someone who, as an adult, dressed in one of the worst, if not the worst stereoty',
 "2019: The year that actual photos and video have somehow ceased to be what you're actually looking at.",
 "Real talk. Who among us hasn't chosen a photo of two people we did not know wearing blackface and a KKK costume for our yearbook page? ",
 "Okay, how about this one? It's a pattern, man. It's not a one off. ",
 'Agreed! Now do Steve King. ',
 "LOL! You're so nice, but NO THANKS!",
 'Reminder that there are far more insidious types of racism in politics than an old photo and all of it, including the guy in the old photo, needs to go.',
 "That's what he did.",
 'No, the implicatio

In [19]:
all_nodes, ner_visuals = get_nodes_from_batch(test_sents)

1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
No Relations Found
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/ste

In [42]:
doc = nlp(test_sents[10])
[(tk,pos) for tk, pos in zip([t.text for t in doc],[t.pos_ for t in doc])]

[('No', 'INTJ'),
 (',', 'PUNCT'),
 ('the', 'DET'),
 ('implication', 'NOUN'),
 (',', 'PUNCT'),
 ('IM', 'PROPN'),
 ('HO', 'PROPN'),
 (',', 'PUNCT'),
 ('is', 'AUX'),
 ('that', 'SCONJ'),
 ('anyone', 'PRON'),
 ('whose', 'DET'),
 ('judgement', 'NOUN'),
 ('allowed', 'VERB'),
 ('them', 'PRON'),
 ('to', 'PART'),
 ('be', 'AUX'),
 ('in', 'ADP'),
 ('that', 'DET'),
 ('photo', 'NOUN'),
 ('can', 'AUX'),
 ('rightfully', 'ADV'),
 ('govern', 'VERB'),
 ('a', 'DET'),
 ('state', 'NOUN'),
 ('that', 'PRON'),
 ('has', 'VERB'),
 ('a', 'DET'),
 ('sizable', 'ADJ'),
 ('(', 'PUNCT'),
 ('or', 'CCONJ'),
 ('really', 'ADV'),
 ('any', 'DET'),
 (')', 'PUNCT'),
 ('population', 'NOUN'),
 ('that', 'PRON'),
 ('holds', 'VERB'),
 ('those', 'DET'),
 ('tropes', 'NOUN'),
 ('to', 'PART'),
 ('be', 'AUX'),
 ('the', 'DET'),
 ('most', 'ADV'),
 ('offensive', 'ADJ'),
 ('there', 'PRON'),
 ('are', 'VERB'),
 ('.', 'PUNCT')]

In [20]:
ner_visuals

,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,t11,t12,t13,t14,t15,t16,t17,t18,t19,t20,t21,t22,t23,t24,t25,t26,t27,t28,t29,t30,t31,t32,t33,t34,t35,t36,t37,t38,t39,t40,t41,t42,t43,t44,t45,t46,t47,t48,t49,t50,t51,t52,t53,t54,t55,t56,t57,t58,t59,t60,t61,t62,t63,t64,t65,t66,t67,t68,t69,t70,t71,t72,t73,t74,t75,t76,t77,t78,t79
0,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,I-Value,O,O,I-Value,I-Value,O,O,O,O,O,O,B-Value,O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,That,is,your,prerogative,if,you,want,to,support,him,",",but,there,are,plenty,of,other,great,people,who,will,fight,for,everyone,'s,rights,who,haven,'t,ever,considered,wearing,blackface,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,O,O,O,B-Identity,O,O,O,O,O,O,O,O,O,O,O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,of,these,dudes,can,ever,believe,they,'re,being,called,on,their,shit,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Everyone,deserves,to,be,given,a,chance,to,change,",",but,how,can,someone,who,",",as,an,adult,",",dressed,in,one,of,the,worst,",",if,not,the,worst,stereoty,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019,:,The,year,that,actual,photos,and,video,have,somehow,ceased,to,be,what,you,'re,actually,looking,at,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,I-Object,O,O,B-Object,I-Object,O,O,B-Object,O,O,O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Real,talk,.,Who,among,us,hasn,'t,chosen,a,photo,of,two,people,we,did,not,know,wearing,blackface,and,a,KKK,costume,for,our,yearbook,page,?,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
all_nodes.loc[all_nodes['pred_rel'] != 'none']

,sid,sentence_text,lefttext,righttext,pred_rel,pred_val
0,7603,"[Steve, King, is, going, to, endorse, Ralph, N...",Steve King,Ralph Northam,Threat,Neutral
0,7952,"[It, 's, not, a, comparison, ., Pointing, out,...",King,racism,Agrees,Negative
1,7952,"[It, 's, not, a, comparison, ., Pointing, out,...",racism,King,Agrees,Negative
0,6155,"[Man, oh, man, ., This, is, real, ., Just, saw...",iTunes,AppleSupport,Is,Neutral
1,6155,"[Man, oh, man, ., This, is, real, ., Just, saw...",AppleSupport,iTunes,Is,Neutral
3,4238,"[Cool, platform, @, Jack, ., ]",.,@,Cooperates,Neutral
4,4238,"[Cool, platform, @, Jack, ., ]",@,,Cooperates,Neutral
5,4238,"[Cool, platform, @, Jack, ., ]",,@,Cooperates,Neutral
2,8782,"[FREE, IDEA, FOR, @, jack, :, Get, rid, of, AL...",@,bots,Present,Neutral
3,8782,"[FREE, IDEA, FOR, @, jack, :, Get, rid, of, AL...",bots,@,Present,Neutral
